# NeuroAD Discovery Engine — end-to-end walkthrough

A reproducible run of the referee on the cached embedding-table contract.
The whole loop is `run_referee(df, claim) -> ClaimCard`:

> probe (naive effect) → gauntlet (5 adversarial tests) → verdict → (survivors only) Claude adjudication + biology bridge → reviewer.

Every cell is import-guarded: if a sibling module (engine / data / Claude layer)
has not landed yet, the cell prints a note instead of raising. The demo runs
fully offline (synthetic harness + Claude template fallback).

In [ ]:
# Make `src/` importable when running from the repo root or notebooks/.
import sys, os
from pathlib import Path
here = Path.cwd()
root = here if (here / 'src').exists() else here.parent
src = str(root / 'src')
if src not in sys.path:
    sys.path.insert(0, src)

from neuroad import contract, calibration
print('contract version:', contract.CONTRACT_VERSION)
print('positioning:', calibration.POSITIONING[:120], '...')

## 1. Load a cohort

`loaders.load` dispatches `synthetic:SURVIVOR`, `synthetic:KILL`, or `oasis`.
If the data layer is not present yet we build a tiny contract-valid table inline
so the rest of the notebook still runs.

In [ ]:
import numpy as np, pandas as pd

def _tiny_cohort(n=120, d=8, seed=0):
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 2, n)
    X = rng.normal(size=(n, d)); X[:, 0] += 1.6 * y; X[:, 1] += 0.9 * y
    f = contract.make_embedding_frame(X)
    f['subject_id'] = [f's{i:03d}' for i in range(n)]
    f['dx'] = pd.Categorical(np.where(y == 1, 'MCI', 'CN'), categories=contract.DX_LEVELS)
    f['conversion'] = pd.array(y, dtype='Int8')
    f['age'] = rng.normal(72, 6, n)
    f['sex'] = pd.Categorical(rng.choice(['M', 'F'], n), categories=contract.SEX_LEVELS)
    f['site'] = pd.Categorical(rng.choice(['S1', 'S2'], n))
    f['scanner'] = pd.Categorical(rng.choice(['A', 'B'], n))
    f['amyloid'] = pd.array(rng.integers(0, 2, n), dtype='Int8')
    f['p_tau217'] = rng.normal(1.0, 0.3, n) + 0.2 * y
    f['gfap'] = rng.normal(120, 30, n)
    f['nfl'] = rng.normal(20, 5, n)
    f['apoe4'] = pd.array(rng.integers(0, 3, n), dtype='Int8')
    contract.validate_table(f)
    return f

try:
    from neuroad.data import loaders
    df = loaders.load('synthetic:SURVIVOR')
    print('loaded synthetic:SURVIVOR from the data layer')
except Exception as exc:
    df = _tiny_cohort()
    print('data layer unavailable (', type(exc).__name__, ') — using inline tiny cohort')

from pprint import pprint
pprint(contract.cohort_summary(df))

## 2. Run the referee

`pipeline.run_referee` needs the core engine (`probe`, `gauntlet`, `scoring`).
If it is not present, we skip — the notebook does not fail the build.

In [ ]:
card = None
try:
    from neuroad import pipeline
    claim = contract.Claim(
        claim_id='nb-1',
        claim_text='MCI patients who convert to AD show a distinct structural signature.',
        target='conversion',
        group_a='MCI converters', group_b='MCI non-converters',
    )
    card = pipeline.run_referee(df, claim)
    print('verdict:', card.verdict.value, '| score:', card.score, '| promoted:', card.promoted)
except Exception as exc:
    print('core engine not available yet (', type(exc).__name__, ':', exc, ') — skipping run')

## 3. Render the claim card

In [ ]:
if card is not None:
    ne = card.naive_effect
    print('=' * 60)
    print('CLAIM:', card.claim.claim_text)
    print('naive effect:', ne.get('metric'), '=', ne.get('value'), '(n=%s)' % ne.get('n'))
    print('-' * 60)
    for t in card.tests:
        dim = contract.GAUNTLET_BY_KEY.get(t.key)
        print('  %-26s %-14s %s' % (dim.label if dim else t.key, t.result.value, t.detail))
    print('-' * 60)
    print('ROBUSTNESS:', card.score, '/100  →  VERDICT:', card.verdict.value.upper())
    if card.biology_hypothesis:
        print('biology:', card.biology_hypothesis)
    for step in card.next_experiment:
        print('  next:', step)
    for c in card.caveats:
        print('  caveat:', c)
    print('narration:', getattr(card, 'narration', ''))
    print('=' * 60)
else:
    print('no card — core engine not present in this checkout')

## 4. Export the artifact

The card serializes through the contract (`to_dict`) — this is exactly what the
offline workbench in `app/` reads from `reports/`.

In [ ]:
import json
if card is not None:
    payload = card.to_dict()
    payload['narration'] = getattr(card, 'narration', '')
    print(json.dumps(payload, indent=2, default=str)[:1200], '...')
else:
    print('nothing to export')